In [6]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

from src.arranger.arranger import arrange_chord_sequence
from src.synth.midi_renderer import *
from src.synth.midi_renderer import synthesize_with_drums
from src.audio.pitch import *
from src.audio.chroma import extract_chroma
from src.harmony.viterbi import suggest_chords_viterbi
from IPython.display import Audio
import librosa
import pretty_midi

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Teste do arranjador

In [7]:
audio_path = '../data/audio-segment1-acima-do-sol-lead.mp3'
TOM='G'
BPM=92
OFFSET=1
y, sr = librosa.load(audio_path)
listen_beats(y, sr, BPM, offset=OFFSET)
time, frequency, confidence = extract_f0_pyin(y, sr)
df_compassos = extract_chroma(
    time,
    frequency,
    confidence,
    BPM,
    tom=TOM,
    conf_thresh=0,
    quantizar_para_tom=True
)

acordes, caminho, funcoes = suggest_chords_viterbi(
  df_compassos, 
  tom_raiz='G', 
  tamanho_bloco=2, 
  permitir_diminutos=False,
)

df_compassos['acorde'] = acordes
df_compassos['funcao'] = df_compassos['acorde'].map(funcoes)

print(df_compassos[['compasso', 'nota_dominante', 'acorde', 'funcao']])


   compasso nota_dominante acorde funcao
0         1              A     Am     SD
1         2              B     Am     SD
2         3              D      G      T
3         4              D      G      T
4         5              A     Am     SD
5         6              A     Am     SD
6         7              G      G      T


In [8]:
atividade_vocal = extract_vocal_activity(
    time,
    frequency,
    confidence,
    BPM,
    OFFSET,
    min_voiced_ratio=0.3
)

print(atividade_vocal)

[np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_]


In [13]:
eventos_midi = arrange_chord_sequence(
    acordes_sugeridos=acordes,
    tom_raiz=TOM,
    atividade_vocal=atividade_vocal,
    bpm=BPM,
    incluir_bateria=True
)

pm = export_events_to_midi(eventos_midi, '../data/midis/output.mid')
audio = synthesize_with_drums(pm, fs=22050, drum_gain=256.0)
offset = int(OFFSET * (60 / BPM) * 22050)

# 2. Aplica o pad no áudio sintetizado
audio_final = np.pad(audio, (offset, 0), mode="constant")

# 3. Igualar os tamanhos de audio_final e y para permitir a soma sem erro de broadcast
tamanho_max = max(len(audio_final), len(y))
audio_final_aligned = np.pad(audio_final, (0, tamanho_max - len(audio_final)))
y_aligned = np.pad(y, (0, tamanho_max - len(y)))

# 4. Mixagem final e reprodução
audio_mix = audio_final_aligned + y_aligned
Audio(audio_mix, rate=22050)

In [6]:

pm = export_events_to_midi(eventos_midi, '../data/midis/output.mid')

# instrumento de bateria deveria vir zerado pelo sintetizador padrão
idx_bateria = next(i for i, inst in enumerate(pm.instruments) if inst.is_drum)
print("synthesize padrão (deve ser tudo zero):", 
      np.abs(pm.instruments[idx_bateria].synthesize(fs=22050)).sum())

audio = synthesize_with_drums(pm, fs=22050)
print("synthesize_with_drums (deve ter energia > 0):", np.abs(audio).sum())

synthesize padrão (deve ser tudo zero): 0.0
synthesize_with_drums (deve ter energia > 0): 15003981.973864546
